## PIIMiddleware
- 用于检测和处理对话中的个人身份信息（Personally Identifiable Information，PII），支持自定义处理策略


In [1]:
from langchain.agents.middleware import PIIMiddleware
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage


load_dotenv(override=True)

model = init_chat_model(
    model = "deepseek:deepseek-v4-flash",
    profile={"max_input_tokens": 128_000}
)

agent = create_agent(
    model = model,
    # pii_type参数[email,credit_card,mac_address,url,ip]
    #四种策略:redact-[REDACTED_EMAIL],mask-****-****-****-5100,hash-<mac_address_hash:568f198f>,block-抛异常
    #可自行自定义,detector=判断函数|字符串正则
    middleware=[
        PIIMiddleware(pii_type="email",strategy="redact"),
        PIIMiddleware(pii_type="credit_card",strategy="mask"),
        PIIMiddleware(pii_type="mac_address",strategy="hash"),
        PIIMiddleware(pii_type="url",strategy="redact"),
        #默认在模型输入前处理
        PIIMiddleware(pii_type="ip",strategy="block",apply_to_input=True),
        PIIMiddleware("api_key", strategy="hash", apply_to_input=True,
detector=r"sk-[a-zA-Z0-9]+")
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("""
    帮我向 156168188@qq.com 发送一封邮件
    同时查看银行卡号： 5105-1051-0510-5100 的余额
    访问 https://localhost:12345
    确认这是不是 MAC地址： 11-11-11-11-11-11
""")]
})

for msg in response["messages"]:
    msg.pretty_print()

try:
    response2 = agent.invoke({
        "messages": [HumanMessage("我的IP地址是 192.168.1.1,帮我ping一下")]
    })
except Exception as e:
    print(f"含有ip信息：{e}")



================================ Human Message =================================


    帮我向 [REDACTED_EMAIL] 发送一封邮件
    同时查看银行卡号： ****-****-****-5100 的余额
    访问 [REDACTED_URL]
    确认这是不是 MAC地址： <mac_address_hash:568f198f>

================================== Ai Message ==================================

抱歉，我无法帮你发送邮件、查询银行卡余额、访问网址或确认 MAC 地址信息。这些操作涉及隐私、安全和外部系统访问，不在我的能力范围内。建议你通过正规渠道办理相关事务。
含有ip信息：Detected 1 instance(s) of ip in text content
